# Tutorial 5 — Advanced Gradient & Quadrature Filters

This tutorial covers **specialized gradient operators** and **quadrature
filters** from the `le_imgproc` module that go beyond the standard Sobel,
Scharr, and Prewitt filters introduced in Tutorial 1.

**Classes covered:**

| Class | Module | Description |
|-------|--------|-------------|
| `SusanGradient` | `le_imgproc` | Smallest Univalue Segment Assimilating Nucleus |
| `RCMGradientColor` | `le_imgproc` | Robust Color Morphological Gradient (3-ch) |
| `RCMGradient` / `_f32` / `_f64` | `le_imgproc` | RCMG for single-channel images |
| `QuadratureG2` / `_f32` / `_f64` | `le_imgproc` | Gaussian 2nd-order quadrature |
| `QuadratureLGF` / `_f32` / `_f64` | `le_imgproc` | Log-Gabor quadrature |
| `QuadratureS` / `_f32` / `_f64` | `le_imgproc` | Phase-based (scale) quadrature |
| `QuadratureSF` / `_f32` / `_f64` | `le_imgproc` | Scale-frequency quadrature |

**Prerequisites:**
- Completed Tutorial 1 (or equivalent)
- Built bindings: `bazel build //libs/...`

**Sections:**

1. Setup & Imports
2. Test Images
3. SUSAN Gradient
4. RCMG Color Gradient
5. RCMG Grayscale Gradient
6. Quadrature Filters — Overview
7. QuadratureG2 — Gaussian Quadrature
8. QuadratureLGF — Log-Gabor Quadrature
9. QuadratureS — Scale Quadrature
10. QuadratureSF — Scale-Frequency Quadrature
11. Quadrature Comparison
12. Comprehension Questions
13. Challenge Exercise

---
## 1. Setup & Imports

In [ ]:
import sys, pathlib

# --- Locate workspace root and add Bazel output dirs to sys.path ---
workspace = pathlib.Path.cwd()
while not (workspace / "MODULE.bazel").exists():
    if workspace == workspace.parent:
        raise RuntimeError("Cannot find LineExtraction workspace root (MODULE.bazel)")
    workspace = workspace.parent

for lib in ["imgproc", "edge", "geometry", "eval", "lsd", "algorithm"]:
    p = workspace / f"bazel-bin/libs/{lib}/python"
    if p.exists():
        sys.path.insert(0, str(p))
    else:
        print(f"Warning: Not found: {p}  \u2014 run: bazel build //libs/{lib}/...")

sys.path.insert(0, str(workspace / "python"))

import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import le_imgproc
from lsfm.data import TestImages

print(f"Workspace: {workspace}")
print(f"le_imgproc loaded: {len([x for x in dir(le_imgproc) if not x.startswith('_')])} symbols")

In [ ]:
# --- Common visualization helpers ---

def show_images(images, titles, cmap="gray", figsize=None):
    """Show a list of images side by side."""
    n = len(images)
    if figsize is None:
        figsize = (4 * n, 4)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_filter_outputs(outputs_dict, figsize=None):
    """Visualize multiple filter output images as a grid."""
    n = len(outputs_dict)
    if figsize is None:
        figsize = (4 * n, 4)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, (title, img) in zip(axes, outputs_dict.items()):
        ax.imshow(img, cmap="gray")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

---
## 2. Test Images

We load a windmill test image (grayscale + color) for demonstrations.

In [ ]:
# Load test images
ti = TestImages(workspace)

# Grayscale image (uint8)
img_gray = cv2.imread(str(ti.windmill()), cv2.IMREAD_GRAYSCALE)
print(f"Grayscale: {img_gray.shape}, dtype={img_gray.dtype}")

# Float64 version for quadrature filters
img_f64 = img_gray.astype(np.float64)

# Color image (uint8, BGR -> RGB for display)
img_color_bgr = cv2.imread(str(ti.windmill()), cv2.IMREAD_COLOR)
img_color_rgb = img_color_bgr[:, :, ::-1].copy()
print(f"Color: {img_color_bgr.shape}, dtype={img_color_bgr.dtype}")

show_images([img_gray, img_color_rgb], ["Grayscale", "Color (RGB)"])

---
## 3. SUSAN Gradient

The **Smallest Univalue Segment Assimilating Nucleus (SUSAN)** operator
detects edges by analyzing brightness similarity within a circular mask.

- Input: **uint8 only**
- `brightness_th` — brightness similarity threshold (default 20)
- `small_kernel` — use 3\u00d73 kernel instead of 37-pixel SUSAN mask
- `max_no` — maximum USAN area; higher = more sensitive (default 2650)

Outputs: `.magnitude()`, `.direction()`, `.gx()`, `.gy()`

In [ ]:
# --- SUSAN gradient with default parameters ---
susan = le_imgproc.SusanGradient()
susan.process(img_gray)

print(f"Magnitude shape: {susan.magnitude().shape}, dtype: {susan.magnitude().dtype}")
print(f"Direction shape: {susan.direction().shape}")

show_filter_outputs({
    "SUSAN Magnitude": susan.magnitude(),
    "SUSAN Direction": susan.direction(),
    "SUSAN Gx": susan.gx(),
    "SUSAN Gy": susan.gy(),
})

In [ ]:
# --- Effect of brightness_th parameter ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, bth in zip(axes, [5, 15, 30, 60]):
    s = le_imgproc.SusanGradient(brightness_th=bth)
    s.process(img_gray)
    ax.imshow(s.magnitude(), cmap="gray")
    ax.set_title(f"brightness_th={bth}")
    ax.axis("off")
plt.suptitle("SUSAN: brightness threshold sweep", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Small vs full SUSAN kernel ---
susan_full = le_imgproc.SusanGradient(brightness_th=20, small_kernel=False)
susan_small = le_imgproc.SusanGradient(brightness_th=20, small_kernel=True)

susan_full.process(img_gray)
susan_small.process(img_gray)

show_images(
    [susan_full.magnitude(), susan_small.magnitude()],
    ["Full kernel (37 pixels)", "Small kernel (3\u00d73)"],
)

---
## 4. RCMGradientColor — Robust Color Morphological Gradient

The **RCMG** operator computes edge strength using morphological
operations in color space. `RCMGradientColor` operates on **3-channel
uint8** images.

- `mask` — structuring element size (3, 5, or 7; default 3)
- `s` — smoothing iterations (default 0)
- `norm` — distance norm (`NormType.L2SQR`, `.L1`, `.L2`, `.LINF`)

Outputs: `.magnitude()`, `.direction()`, `.gx()`, `.gy()`

In [ ]:
# --- RCMG color gradient ---
rcmg_color = le_imgproc.RCMGradientColor(mask=3, s=0)
rcmg_color.process(img_color_bgr)

print(f"Magnitude: {rcmg_color.magnitude().shape}, dtype: {rcmg_color.magnitude().dtype}")

show_filter_outputs({
    "RCMG Color Magnitude": rcmg_color.magnitude(),
    "RCMG Color Direction": rcmg_color.direction(),
})

In [ ]:
# --- Mask size comparison on color image ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, mask_size in zip(axes, [3, 5, 7]):
    rc = le_imgproc.RCMGradientColor(mask=mask_size)
    rc.process(img_color_bgr)
    ax.imshow(rc.magnitude(), cmap="gray")
    ax.set_title(f"mask={mask_size}")
    ax.axis("off")
plt.suptitle("RCMGradientColor: mask size sweep", fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. RCMGradient — Grayscale RCMG

`RCMGradient` is the single-channel version with typed variants:
- `RCMGradient` — uint8 (default)
- `RCMGradient_f32` — float32
- `RCMGradient_f64` — float64

Same parameters as `RCMGradientColor` (`mask`, `s`, `norm`).

In [ ]:
# --- Compare RCMG color vs grayscale ---
rcmg_gray = le_imgproc.RCMGradient(mask=3)
rcmg_gray.process(img_gray)

rcmg_f64 = le_imgproc.RCMGradient_f64(mask=3)
rcmg_f64.process(img_f64)

show_images(
    [rcmg_color.magnitude(), rcmg_gray.magnitude(), rcmg_f64.magnitude()],
    ["RCMG Color (3-ch uint8)", "RCMG Gray (uint8)", "RCMG Gray (float64)"],
)

---
## 6. Quadrature Filters — Overview

Quadrature filters decompose an image into **even** (symmetric) and
**odd** (antisymmetric) components, enabling computation of:

- **Phase** — local structure type (line, edge, ridge, etc.)
- **Energy** — orientation-independent edge/feature strength
- **Direction** — local gradient orientation
- **Laplacian** — second-order information

All four quadrature classes share a **common interface:**

| Method | Description |
|--------|-------------|
| `.process(img)` | Compute all filter responses |
| `.even()` | Even (symmetric) response |
| `.odd()` | Odd (antisymmetric) response magnitude |
| `.odd_xy()` | Odd response as (x, y) components |
| `.oddx()`, `.oddy()` | Individual odd components |
| `.direction()` | Local orientation |
| `.energy()` | Phase-independent energy |
| `.phase()` | Local phase |
| `.laplace()` | Laplacian response |

Each class has 3 presets: default (uint8), `_f32`, `_f64`.

| Class | Approach | Key Parameters |
|-------|----------|----------------|
| `QuadratureG2` | Gaussian 2nd-order | `kernel_size`, `kernel_spacing` |
| `QuadratureLGF` | Log-Gabor | `wave_length`, `sigma_onf` |
| `QuadratureS` | Phase-based (scale) | `scale`, `muls`, `kernel_size` |
| `QuadratureSF` | Scale-frequency | `scale`, `muls`, `kernel_spacing` |

In [ ]:
# Helper: visualize all quadrature outputs for a given filter
def show_quadrature(name, qf):
    """Show the key outputs of a quadrature filter."""
    outputs = {
        "Even": qf.even(),
        "Odd": qf.odd(),
        "Energy": qf.energy(),
        "Phase": qf.phase(),
        "Direction": qf.direction(),
        "Laplacian": qf.laplace(),
    }
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, (title, img) in zip(axes.flat, outputs.items()):
        ax.imshow(img, cmap="gray")
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(name, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## 7. QuadratureG2 — Gaussian Quadrature

Uses second-order Gaussian derivatives to build even/odd filter pairs.

**Parameters:**
- `kernel_size` — filter kernel size (default 9)
- `kernel_spacing` — spacing between samples (default 0.782)

**Extra methods:** `.steer(angle)`, `.steer_xy(angle)`, `.energy_phase()`,
`.kernel()` — the raw filter kernel.

In [ ]:
# --- QuadratureG2 with default parameters ---
qg2 = le_imgproc.QuadratureG2_f64()
qg2.process(img_f64)

print(f"Even shape: {qg2.even().shape}, dtype: {qg2.even().dtype}")
print(f"Energy range: [{qg2.energy().min():.2f}, {qg2.energy().max():.2f}]")

show_quadrature("QuadratureG2 (kernel_size=9, spacing=0.782)", qg2)

In [ ]:
# --- G2-specific: steerable filter and energy-phase decomposition ---
angles = [0, 45, 90, 135]
fig, axes = plt.subplots(1, len(angles), figsize=(16, 4))
for ax, angle in zip(axes, angles):
    # steer() expects a full orientation image — create a constant one
    theta_img = np.full_like(img_f64, np.radians(angle))
    g2, h2 = qg2.steer(theta_img)
    ax.imshow(g2, cmap="gray")
    ax.set_title(f"Steered {angle}°")
    ax.axis("off")
plt.suptitle("QuadratureG2: steerable filter at different angles", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Kernel spacing effect ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, spacing in zip(axes, [0.5, 0.782, 1.2]):
    q = le_imgproc.QuadratureG2_f64(kernel_size=9, kernel_spacing=spacing)
    q.process(img_f64)
    ax.imshow(q.energy(), cmap="gray")
    ax.set_title(f"spacing={spacing}")
    ax.axis("off")
plt.suptitle("QuadratureG2: kernel spacing sweep", fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. QuadratureLGF — Log-Gabor Quadrature

Log-Gabor filters have a Gaussian transfer function on a logarithmic
frequency axis. This avoids the DC component problem of standard Gabor
filters.

**Parameters:**
- `wave_length` — center wavelength (default 5)
- `sigma_onf` — bandwidth parameter (default 0.55)

In [ ]:
# --- QuadratureLGF with default parameters ---
qlgf = le_imgproc.QuadratureLGF_f64()
qlgf.process(img_f64)

print(f"Energy range: [{qlgf.energy().min():.2f}, {qlgf.energy().max():.2f}]")
show_quadrature("QuadratureLGF (wave_length=5, sigma_onf=0.55)", qlgf)

In [ ]:
# --- Wave length sweep ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, wl in zip(axes, [3, 5, 8, 12]):
    q = le_imgproc.QuadratureLGF_f64(wave_length=wl)
    q.process(img_f64)
    ax.imshow(q.energy(), cmap="gray")
    ax.set_title(f"wave_length={wl}")
    ax.axis("off")
plt.suptitle("QuadratureLGF: wavelength sweep", fontsize=13)
plt.tight_layout()
plt.show()

---
## 9. QuadratureS — Scale Quadrature

Phase-based quadrature that operates at multiple scales, controlled by the
`scale` and `muls` (number of scales) parameters.

**Parameters:**
- `scale` — base scale (default 1)
- `muls` — number of scales to combine (default 3)
- `kernel_size` — filter size (default 5)
- `kernel_spacing` — sample spacing (default 1)
- `kernel_scale` — scale factor (default 1)

In [ ]:
# --- QuadratureS with default parameters ---
qs = le_imgproc.QuadratureS_f64()
qs.process(img_f64)

print(f"Energy range: [{qs.energy().min():.2f}, {qs.energy().max():.2f}]")
show_quadrature("QuadratureS (scale=1, muls=3)", qs)

In [ ]:
# --- Scale and muls sweep ---
configs = [(1, 2), (1, 3), (2, 3), (3, 4)]
fig, axes = plt.subplots(1, len(configs), figsize=(16, 4))
for ax, (scale, muls) in zip(axes, configs):
    q = le_imgproc.QuadratureS_f64(scale=scale, muls=muls)
    q.process(img_f64)
    ax.imshow(q.energy(), cmap="gray")
    ax.set_title(f"scale={scale}, muls={muls}")
    ax.axis("off")
plt.suptitle("QuadratureS: scale and muls sweep", fontsize=13)
plt.tight_layout()
plt.show()

---
## 10. QuadratureSF — Scale-Frequency Quadrature

A frequency-domain quadrature filter combining scale and frequency
decomposition.

**Parameters:**
- `scale` — base scale (default 1)
- `muls` — number of scales (default 2)
- `kernel_spacing` — spacing (default 1)

In [ ]:
# --- QuadratureSF with default parameters ---
qsf = le_imgproc.QuadratureSF_f64()
qsf.process(img_f64)

print(f"Energy range: [{qsf.energy().min():.2f}, {qsf.energy().max():.2f}]")
show_quadrature("QuadratureSF (scale=1, muls=2)", qsf)

---
## 11. Quadrature Comparison

Let's compare the energy and phase outputs of all four quadrature filters
side by side.

In [ ]:
# --- Energy comparison ---
filters = {
    "G2": qg2,
    "LGF": qlgf,
    "S": qs,
    "SF": qsf,
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, (name, qf) in enumerate(filters.items()):
    axes[0, col].imshow(qf.energy(), cmap="gray")
    axes[0, col].set_title(f"{name} Energy")
    axes[0, col].axis("off")

    axes[1, col].imshow(qf.phase(), cmap="hsv")
    axes[1, col].set_title(f"{name} Phase")
    axes[1, col].axis("off")

plt.suptitle("Quadrature filter comparison: Energy (top) vs Phase (bottom)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Odd components comparison ---
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, (name, qf) in enumerate(filters.items()):
    axes[0, col].imshow(qf.oddx(), cmap="gray")
    axes[0, col].set_title(f"{name} Odd-X")
    axes[0, col].axis("off")

    axes[1, col].imshow(qf.oddy(), cmap="gray")
    axes[1, col].set_title(f"{name} Odd-Y")
    axes[1, col].axis("off")

plt.suptitle("Odd components: X (top) vs Y (bottom)", fontsize=14)
plt.tight_layout()
plt.show()

**Key observations:**

- **G2** gives the sharpest localization, controlled by `kernel_spacing`
- **LGF** has the best frequency selectivity via `wave_length`
- **S** combines multiple scales for robust multi-resolution response
- **SF** is similar to S but operates in the frequency domain

Choose based on your application:
- Steerable analysis → **QuadratureG2**
- Narrow-band frequency selection → **QuadratureLGF**
- Multi-scale robustness → **QuadratureS** or **QuadratureSF**

---
## 12. Comprehension Questions

1. What input type does `SusanGradient` require? Why?
2. How does `RCMGradientColor` differ from `RCMGradient`?
3. What is the difference between `.energy()` and `.phase()` in quadrature filters?
4. When would you use `QuadratureG2.steer()` versus `.energy()`?
5. How does increasing `wave_length` in `QuadratureLGF` affect the result?

---
## 13. Challenge Exercise

**Task:** Build a combined edge strength map.

1. Compute SUSAN gradient, RCMG gradient, and QuadratureG2 energy
   on the same grayscale image
2. Normalize each to [0, 1] range
3. Create a weighted combination: `0.3 * susan + 0.3 * rcmg + 0.4 * energy`
4. Display individual and combined results
5. Experiment with different weights and discuss which filter contributes
   most to thin vs thick edge structures

In [ ]:
# Your code here:
# ...


---
## Summary

| Component | Type | Key Parameters |
|-----------|------|----------------|
| `SusanGradient` | Edge operator | `brightness_th`, `small_kernel` |
| `RCMGradientColor` | Color edge operator | `mask`, `s`, `norm` |
| `RCMGradient` | Grayscale edge operator | `mask`, `s`, `norm` |
| `QuadratureG2` | Gaussian quadrature | `kernel_size`, `kernel_spacing` |
| `QuadratureLGF` | Log-Gabor quadrature | `wave_length`, `sigma_onf` |
| `QuadratureS` | Scale quadrature | `scale`, `muls`, `kernel_size` |
| `QuadratureSF` | Scale-frequency quadrature | `scale`, `muls`, `kernel_spacing` |

### Next Steps

- **Tutorial 6**: Image operators (noise, blur, geometric transforms, pipelines)
- **Tutorial 7**: 3D geometry (Line3, Plane, Pose, Camera) with Rerun visualization
- Combine quadrature phase with edge detection for structure-aware line analysis